# UCS420: Cognitive Computing
## Assignment 7 — NLP-I

**Name:** Parth Khurana  
**Roll Number:** 1024170354  
**Batch:** 3Q33

**Important:** The PDF requires the generated IT-support-ticket dataset from `UCS420_LA9_Starter.ipynb` and says not to modify `generate_dataset()` or `process_ticket()`. Run that starter notebook first, then run these analysis cells. The starter's exact return schema is not included in the assignment PDF, so inspect the printed output and map its fields as needed.


## Setup

In [ ]:
import re, string, math
from collections import Counter
import pandas as pd
import nltk
import matplotlib.pyplot as plt
from nltk.tokenize import word_tokenize, sent_tokenize, RegexpTokenizer
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer
for resource in ["punkt","punkt_tab","stopwords","averaged_perceptron_tagger","averaged_perceptron_tagger_eng","wordnet","omw-1.4"]:
    nltk.download(resource, quiet=True)
stop_words = set(stopwords.words("english"))

## 1. Preprocessing table

In [ ]:
# Do not edit the provided generate_dataset() or process_ticket() functions.
if "generate_dataset" not in globals() or "process_ticket" not in globals():
    print("Run UCS420_LA9_Starter.ipynb first. Then rerun this cell.")
else:
    generated = generate_dataset(int(roll))
    print("Generated dataset type:", type(generated))
    print("Preview:", str(generated)[:1000])
    print("process_ticket is available:", callable(process_ticket))

In [ ]:
# Set raw_tickets to the raw ticket strings produced by the starter notebook.
# This explicit step avoids guessing the starter notebook's data structure.
try:
    raw_tickets
except NameError:
    raw_tickets = None

if raw_tickets is None:
    print("Set raw_tickets = [raw ticket strings from the starter dataset] before continuing.")
else:
    processed = [process_ticket(t) for t in raw_tickets]
    print("Example process_ticket output:", processed[0])
    def get_tokens(value):
        if isinstance(value, dict):
            for k in ["final_tokens","tokens","clean_tokens"]:
                if k in value: return get_tokens(value[k])
            return []
        if isinstance(value, (list,tuple)): return [str(x) for x in value]
        if isinstance(value, str): return word_tokenize(value)
        return []
    rows=[]
    for raw, result in zip(raw_tickets, processed):
        toks=word_tokenize(raw)
        punct=[t for t in toks if t and all(c in string.punctuation for c in t)]
        sw=[t for t in toks if t.lower() in stop_words]
        final=get_tokens(result)
        rows.append({"ticket":raw,"number_of_sentences":len(sent_tokenize(raw)),
                     "number_of_tokens":len(toks),"number_of_punctuation_tokens":len(punct),
                     "number_of_stopwords":len(sw),"number_of_unique_tokens":len(set(t.lower() for t in toks)),
                     "number_of_final_tokens":len(final)})
    table=pd.DataFrame(rows)
    totals={"ticket":"TOTAL"}
    for col in ["number_of_sentences","number_of_tokens","number_of_punctuation_tokens","number_of_stopwords","number_of_final_tokens"]:
        totals[col]=table[col].sum()
    totals["number_of_unique_tokens"]=""
    display(pd.concat([table,pd.DataFrame([totals])],ignore_index=True))
    chosen=raw_tickets[0]; chosen_tokens=word_tokenize(chosen)
    print("Selected ticket:",chosen)
    print("Actual tokens:",chosen_tokens)
    print("Punctuation tokens:",[t for t in chosen_tokens if t and all(c in string.punctuation for c in t)])
    print("Stopwords:",[t for t in chosen_tokens if t.lower() in stop_words])
    print("Final tokens:",processed[0])

## 2. Regex extraction and replacement

In [ ]:
TEST_LINES = [
    "Contact helpdesk@example.com at https://support.example.com or call 9876543210.",
    "The update is version 2.0.1 and measured 95.5 units.",
    "Wi-Fi connection failed; Please Restart the Router."
]
texts = raw_tickets if 'raw_tickets' in globals() and raw_tickets else TEST_LINES
joined = "\n".join(texts)
print("Words >5 letters:", re.findall(r"\b[A-Za-z]{6,}\b", joined))
print("Numbers, decimals, versions:", re.findall(r"\b\d+(?:\.\d+)+|\b\d+\b", joined))
print("Capitalized words:", re.findall(r"\b[A-Z][a-zA-Z]*\b", joined))
print("Words beginning with vowel:", re.findall(r"\b[AEIOUaeiou][A-Za-z]*\b", joined))
for pattern,repl in [
    (r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b","<EMAIL>"),
    (r"https?://\S+|www\.\S+","<URL>"),
    (r"(?<!\w)\+?\d[\d ()-]{7,}\d(?!\w)","<PHONE>")
]:
    joined = re.sub(pattern,repl,joined)
print("After replacements:\n",joined[:2000])

## 3. Custom tokenization

In [ ]:
pattern = r"https?://[^\s]+|www\.[^\s]+|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}|\+?\d[\d ()-]{7,}\d|\d+(?:\.\d+)+|[A-Za-z]+(?:['’][A-Za-z]+)?(?:-[A-Za-z]+(?:['’][A-Za-z]+)?)*|\d+|[^\w\s]"
my_tokenizer = RegexpTokenizer(pattern)
for line in TEST_LINES:
    print("\nTEXT:",line)
    print("Custom:",my_tokenizer.tokenize(line))
    print("word_tokenize:",word_tokenize(line))
print("Custom tokenizer aims to keep contractions, hyphenated words, decimals, versions, emails, URLs and phone numbers together. Compare the printed outputs for three concrete differences.")

## 4. Stemming and lemmatization

In [ ]:
porter, lancaster = PorterStemmer(), LancasterStemmer()
regexp_stemmer = RegexpStemmer("ing$", min=5)
lemmatizer = WordNetLemmatizer()
def wn_pos(tag):
    return wordnet.ADJ if tag.startswith("J") else wordnet.VERB if tag.startswith("V") else wordnet.ADV if tag.startswith("R") else wordnet.NOUN
if 'raw_tickets' in globals() and raw_tickets:
    # Reuse final tokens generated by process_ticket; helper supports common return structures.
    words=sorted(set(w.lower() for result in processed for w in get_tokens(result) if w.isalpha()))
else:
    words=sorted(set(w.lower() for w in my_tokenizer.tokenize("\n".join(TEST_LINES)) if w.isalpha()))
tags=dict(nltk.pos_tag(words))
rows=[]
for w in words:
    vals={"word":w,"Porter":porter.stem(w),"Lancaster":lancaster.stem(w),
          "Regexp":regexp_stemmer.stem(w),"WordNet lemma (POS-aware)":lemmatizer.lemmatize(w,wn_pos(tags[w]))}
    if len(set(vals.values()))>1: rows.append(vals)
display(pd.DataFrame(rows))
print("Regexp suffix justification: 'ing' is a common verb ending. Check that at least two words in your own dataset end in 'ing'; if not, choose a suffix occurring in at least two dataset words and update the pattern.")

## 5. Corpus statistics

In [ ]:
if 'raw_tickets' in globals() and raw_tickets:
    S1=[t.lower() for raw in raw_tickets for t in word_tokenize(raw)]
    S2=[t for t in S1 if not (t and all(c in string.punctuation for c in t))]
    S3=[w for result in processed for w in get_tokens(result)]
else:
    S1=[w.lower() for w in my_tokenizer.tokenize("\n".join(TEST_LINES))]
    S2=[w for w in S1 if not (w and all(c in string.punctuation for c in w))]
    S3=[w for w in S2 if w not in stop_words]
S4=[porter.stem(w) for w in S3]
def stats(seq): return {"total tokens":len(seq),"unique tokens":len(set(seq)),"TTR":len(set(seq))/len(seq) if seq else 0}
display(pd.DataFrame({"S1 all tokens":stats(S1),"S2 without punctuation":stats(S2),"S3 final tokens":stats(S3),"S4 Porter stemmed":stats(S4)}).T)
print("Top 10 S1:",Counter(S1).most_common(10))
print("Top 10 S3:",Counter(S3).most_common(10))
print("Top 5 bigrams S1:",Counter(zip(S1,S1[1:])).most_common(5))
print("Top 5 bigrams S3:",Counter(zip(S3,S3[1:])).most_common(5))
freq=sorted(Counter(S3).values(),reverse=True)
if freq:
    plt.figure(figsize=(7,5))
    plt.plot([math.log(i) for i in range(1,len(freq)+1)],[math.log(f) for f in freq],"o")
    plt.xlabel("log(rank)"); plt.ylabel("log(frequency)"); plt.title("S3 log(rank) vs log(frequency)"); plt.grid(True); plt.show()

## 6. Interpretation

**(a) TTR from S1 to S4:** TTR is unique-token count divided by total-token count. Removing punctuation/stopwords and stemming alters both counts; stemming often merges word forms and can reduce unique vocabulary. The exact change depends on the generated dataset.

**(b) Tokenizer example:** The custom tokenizer is designed to keep `Wi-Fi` as one token, while the standard tokenizer may split hyphenated forms differently. Use the printed outputs for your actual example.

**(c) Undesirable stemming:** A rule-based stemmer can truncate a word into a form that is not a dictionary word.

**(d) POS-aware lemmatization:** A verb such as `running` can be lemmatized to `run` when tagged as a verb; POS information helps preserve the intended base form.